**Homework Week 7**

2/27/26

In [12]:
## Set-up

# Imports
import pandas as pd
import seaborn as sns
import plotly.express as px
import numpy as np
import zipfile
import os
from google.colab import files
from datetime import datetime

# Reading in atus files in zipped folder
# uploaded = files.upload()
with zipfile.ZipFile("FinalProjectFiles.zip", 'r') as zip_ref:
    zip_ref.extractall("FinalProjectFiles")

# All atus filepaths
folder_path = "FinalProjectFiles/FinalProjectFiles"
files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith(".dat")]

print(len(files))
print(files)

40
['FinalProjectFiles/FinalProjectFiles/atusrost_2011.dat', 'FinalProjectFiles/FinalProjectFiles/atusact_2013.dat', 'FinalProjectFiles/FinalProjectFiles/atusrost_2017.dat', 'FinalProjectFiles/FinalProjectFiles/atusrost_2013.dat', 'FinalProjectFiles/FinalProjectFiles/atusrost_2015.dat', 'FinalProjectFiles/FinalProjectFiles/atusact_2011.dat', 'FinalProjectFiles/FinalProjectFiles/atusact_2017.dat', 'FinalProjectFiles/FinalProjectFiles/atusact_2014.dat', 'FinalProjectFiles/FinalProjectFiles/atuswho_2018.dat', 'FinalProjectFiles/FinalProjectFiles/atuswho_2017.dat', 'FinalProjectFiles/FinalProjectFiles/atuswho_2019.dat', 'FinalProjectFiles/FinalProjectFiles/atusresp_2012.dat', 'FinalProjectFiles/FinalProjectFiles/atusact_2019.dat', 'FinalProjectFiles/FinalProjectFiles/atusrost_2012.dat', 'FinalProjectFiles/FinalProjectFiles/atusrost_2016.dat', 'FinalProjectFiles/FinalProjectFiles/atusact_2018.dat', 'FinalProjectFiles/FinalProjectFiles/atusresp_2014.dat', 'FinalProjectFiles/FinalProjectFiles

In [13]:
## Reading in data

# Empty lists to store datasets
atusact_files = []
atusresp_files = []
atusrost_files = []
atuswho_files = []

# Separating files into the four dataset categories
for f in files:
    filename = os.path.basename(f)
    if filename.startswith("atusact_"):
        atusact_files.append(f)
    elif filename.startswith("atusresp_"):
        atusresp_files.append(f)
    elif filename.startswith("atusrost_"):
        atusrost_files.append(f)
    elif filename.startswith("atuswho_"):
        atuswho_files.append(f)

print(len(atusact_files), len(atusresp_files), len(atusrost_files), len(atuswho_files))

# Function to stack datasets
def stack_atus_files(file_list):
    df_list = []
    for file_path in sorted(file_list):
        filename = os.path.basename(file_path)
        year = filename[-8:-4]
        df = pd.read_csv(file_path)
        df["year"] = int(year)
        df_list.append(df)
    stacked_df = pd.concat(df_list, ignore_index = True, sort = False) # keeps all columns
    return stacked_df

# Calling function to stack datasets
atusact_all = stack_atus_files(atusact_files)
atusresp_all = stack_atus_files(atusresp_files)
atusrost_all = stack_atus_files(atusrost_files)
atuswho_all = stack_atus_files(atuswho_files)

# Checking stacked datasets
print(atusact_all.shape)
print(atusresp_all.shape)
print(atusrost_all.shape)
print(atuswho_all.shape)

10 10 10 10
(2149915, 32)
(111808, 178)
(302014, 9)
(2762744, 6)


In [14]:
## Keeping relevant columns in datasets

# Activity dataset
atusact_all.columns
atusact_all = atusact_all[["TUCASEID", "TUTIER1CODE", "TRTIER2", "TUSTARTTIM", "TUSTOPTIME", "year"]]
atusact_all.head(10)

# Respondent dataset
atusresp_all.columns
atusresp_all = atusresp_all[["TUCASEID", "TUDIARYDAY", "TUDIARYDATE", "TELFS", "TRERNWA", "year"]]
atusresp_all.head(10)

# Roster dataset
atusrost_all.columns
atusrost_all = atusrost_all[["TUCASEID", "TERRP", "TEAGE", "TESEX", "year"]]
atusrost_all.head(10)

# Who dataset
atuswho_all.columns
atuswho_all = atuswho_all[["TUCASEID", "TUWHO_CODE", "year"]]
atuswho_all.head(10)

,TUCASEID,TUWHO_CODE,year
0,20100101100019,-1,2010
1,20100101100019,-1,2010
2,20100101100019,18,2010
3,20100101100019,18,2010
4,20100101100019,59,2010
5,20100101100019,61,2010
6,20100101100019,18,2010
7,20100101100019,18,2010
8,20100101100019,18,2010
9,20100101100019,18,2010


In [15]:
## Activity dataset cleaning

# Converting start and stop time variables to numeric
atusact_all["TUSTARTTIM"] = pd.to_datetime(atusact_all["TUSTARTTIM"], format = '%H:%M:%S', errors = 'coerce')
atusact_all["TUSTOPTIME"] = pd.to_datetime(atusact_all["TUSTOPTIME"],  format = '%H:%M:%S', errors = 'coerce')

# Converting start and stop times to minutes since midnight
atusact_all["start_minute"] = atusact_all["TUSTARTTIM"].dt.hour*60 + atusact_all["TUSTARTTIM"].dt.minute
atusact_all["stop_minute"]  = atusact_all["TUSTOPTIME"].dt.hour*60 + atusact_all["TUSTOPTIME"].dt.minute

# Adding column for total activity duration
atusact_all["duration"] = atusact_all["stop_minute"] - atusact_all["start_minute"]
atusact_all.loc[atusact_all["duration"] < 0, "duration"] += 24*60

# Adding child care and household tasks indicator columns
atusact_all["child_care"] = atusact_all["TRTIER2"].isin([301, 302, 303]).astype(int)
atusact_all["household_task"] = (atusact_all["TUTIER1CODE"] == 2).astype(int)

# Dropping unnecessary columns
# atusact_all = atusact_all.drop(columns = ["start_minute", "stop_minute", "TUSTARTTIM", "TUSTOPTIME", "TUTIER1CODE", "TRTIER2"])

# Collapsing to a single row per main respondent
atusact_final = (
    atusact_all
    .groupby(["TUCASEID", "year"], as_index = False)
    .agg(
        child_care_duration = ("duration", lambda x: np.sqrt(x[atusact_all.loc[x.index, "child_care"] == 1].sum())),
        household_task_duration = ("duration", lambda x: np.sqrt(x[atusact_all.loc[x.index, "household_task"] == 1].sum()))
    )
)

print(atusact_final.shape)
print(atusact_final.head(20))

(111808, 4)
          TUCASEID  year  child_care_duration  household_task_duration
0   20100101100019  2010             0.000000                 0.000000
1   20100101100020  2010             0.000000                14.317821
2   20100101100045  2010             6.782330                 7.745967
3   20100101100050  2010             0.000000                15.811388
4   20100101100053  2010             0.000000                 4.472136
5   20100101100087  2010             0.000000                 3.872983
6   20100101100095  2010             0.000000                 0.000000
7   20100101100098  2010             0.000000                 5.000000
8   20100101100117  2010             0.000000                 1.000000
9   20100101100119  2010             0.000000                 0.000000
10  20100101100174  2010             0.000000                 5.477226
11  20100101100175  2010             5.477226                 7.745967
12  20100101100501  2010             0.000000                10.2

In [16]:
## Respondent dataset cleaning

# Exploring missing income values
((atusresp_all["TRERNWA"] == -1)).sum() # 51,842 respondents have missing income
((atusresp_all["TRERNWA"] == -1) & ((atusresp_all["TELFS"] == 1) | (atusresp_all["TELFS"] == 2))).sum() # 7,605 employed respondents have missing income
(((atusresp_all["TELFS"] == 1) | (atusresp_all["TELFS"] == 2))).sum() # 67,571 respondents are employed
# 11.25% of employed people are missing income

# Converting missing income values to NA
atusresp_all.loc[atusresp_all["TRERNWA"] == -1, "TRERNWA"] = np.nan

# Imputing 0 income for unemployed people
atusresp_all.loc[(~atusresp_all["TELFS"].isin([1, 2])) & atusresp_all["TRERNWA"].isna(), "TRERNWA"] = 0

# Adding employed indicator
atusresp_all.loc[:, "employed"] = atusresp_all["TELFS"].isin([1, 2]).astype(int)

# Converting diary date variable to date type
atusresp_all["TUDIARYDATE"] = pd.to_datetime(atusresp_all["TUDIARYDATE"].astype(str), format = "%Y%m%d")

# Calculating annual income
atusresp_all.loc[:, "annual_income"] = atusresp_all["TRERNWA"] * 26

# Dropping unnecessary columns
atusresp_final = atusresp_all.drop(columns = ["TRERNWA", "TELFS"])

print(atusresp_final.shape)
print(atusresp_final.head(20))

(111808, 6)
          TUCASEID  TUDIARYDAY TUDIARYDATE  year  employed  annual_income
0   20100101100019           1  2010-01-24  2010         1      2142400.0
1   20100101100020           1  2010-01-31  2010         1      1515592.0
2   20100101100045           3  2010-01-26  2010         1      1560000.0
3   20100101100050           5  2010-01-28  2010         0            0.0
4   20100101100053           1  2010-01-24  2010         0            0.0
5   20100101100087           6  2010-01-29  2010         0            0.0
6   20100101100095           1  2010-01-24  2010         0            0.0
7   20100101100098           4  2010-01-27  2010         0            0.0
8   20100101100117           6  2010-01-29  2010         0            0.0
9   20100101100119           7  2010-01-30  2010         0            0.0
10  20100101100174           1  2010-01-24  2010         0            0.0
11  20100101100175           1  2010-01-31  2010         0            0.0
12  20100101100501        

In [17]:
## Merging respondent and activity datasets

# Merge
atus_actresp = pd.merge(atusresp_final, atusact_final, on = ["TUCASEID", "year"], how = "left")

print(atus_actresp.shape)
atus_actresp.head(10)


(111808, 8)


,TUCASEID,TUDIARYDAY,TUDIARYDATE,year,employed,annual_income,child_care_duration,household_task_duration
0,20100101100019,1,2010-01-24,2010,1,2142400.0,0.00000,0.000000
1,20100101100020,1,2010-01-31,2010,1,1515592.0,0.00000,14.317821
2,20100101100045,3,2010-01-26,2010,1,1560000.0,6.78233,7.745967
3,20100101100050,5,2010-01-28,2010,0,0.0,0.00000,15.811388
4,20100101100053,1,2010-01-24,2010,0,0.0,0.00000,4.472136
5,20100101100087,6,2010-01-29,2010,0,0.0,0.00000,3.872983
6,20100101100095,1,2010-01-24,2010,0,0.0,0.00000,0.000000
7,20100101100098,4,2010-01-27,2010,0,0.0,0.00000,5.000000
8,20100101100117,6,2010-01-29,2010,0,0.0,0.00000,1.000000
9,20100101100119,7,2010-01-30,2010,0,0.0,0.00000,0.000000


In [19]:
## Plot 1

# Filter for employed respondents
atus_employed = atus_actresp[atus_actresp["employed"] == 1].copy()

# Create a total unpaid minutes column
atus_employed["total_unpaid"] = atus_employed["child_care_duration"] + atus_employed["household_task_duration"]

# Random sampling to avoid overplotting
atus_sampled = atus_employed.sample(n=1000, random_state=42)

# Day of week column
day_map = {1: "Sunday", 2: "Monday", 3: "Tuesday", 4: "Wednesday", 5: "Thursday", 6: "Friday", 7: "Saturday"}
atus_sampled["day_of_week"] = atus_sampled["TUDIARYDAY"].map(day_map)

# Split numeric and NA income before plotting
numeric_income = atus_sampled[atus_sampled["annual_income"].notna()]
na_income = atus_sampled[atus_sampled["annual_income"].isna()]

# Custom color scale
custom_scale = [[0, "#E7C5DB"], [1, "#65294F"]]

# Scatter plot for numeric incomes (gradient)
fig = px.scatter(
    numeric_income,
    x="TUDIARYDATE",
    y="total_unpaid",
    color="annual_income",
    color_continuous_scale=custom_scale,
    hover_data=[
        "TUDIARYDATE",
        "day_of_week",
        "child_care_duration",
        "household_task_duration",
        "total_unpaid",
        "annual_income"
    ],
    labels={
        "TUDIARYDATE": "Diary Date",
        "day_of_week": "Day of Week",
        "annual_income": "Annual Income",
        "total_unpaid": "Total Unpaid Minutes",
        "child_care_duration": "Child Care Duration",
        "household_task_duration": "Household Task Duration"
    },
    title="Unpaid Work Time (Child Care, Household Tasks) in 24 Hours for Employed Respondents"
)

# Remove transparency for numeric-income points
fig.update_traces(marker=dict(opacity=1, size=6))

# Scatter for NA income (blue semi-transparent points)
fig.add_scatter(
    x=na_income["TUDIARYDATE"],
    y=na_income["total_unpaid"],
    mode="markers",
    marker=dict(color="#005085", size=6, opacity=0.5),
    hoverinfo="text",
    text=[
        f"Diary Date={row.TUDIARYDATE}<br>Day={row.day_of_week}<br>"
        f"Child Care={row.child_care_duration}<br>Household={row.household_task_duration}<br>"
        f"Total Unpaid={row.total_unpaid}<br>Income=NA"
        for row in na_income.itertuples()
    ],
    name="Income NA"
)

# Determine full x-axis range
full_start = atus_sampled["TUDIARYDATE"].min()
full_end = atus_sampled["TUDIARYDATE"].max()

# Add "All" button first
all_button = dict(
    label="All",
    method="relayout",
    args=[{"xaxis.range": [full_start, full_end]}]
)

# Add updatemenus for year buttons
years = list(range(2010, 2020))
buttons = [all_button]
for y in years:
    start = datetime(y, 1, 1)
    end = datetime(y, 12, 31)
    buttons.append(
        dict(
            label=str(y),
            method="relayout",
            args=[{"xaxis.range": [start, end]}]
        )
    )

# Update layout with buttons and range slider
fig.update_layout(
    margin=dict(b=120),
    xaxis=dict(
        title=dict(text="Diary Date", standoff=120),
        rangeslider=dict(visible=True, bgcolor="lightgrey"),
        type="date",
        showgrid=True,
        gridcolor="lightgrey",
        gridwidth=1,
        zeroline=False
    ),
    yaxis=dict(
        title="Unpaid Sqrt Minutes per Day",
        autorange=True,
        showgrid=True,
        gridcolor="lightgrey",
        gridwidth=1,
        zeroline=False
    ),
    dragmode="zoom",
    legend=dict(
        y=-0.05,
        yanchor="middle",
        x=1.01,
        xanchor="left"
    ),
    paper_bgcolor="white",
    plot_bgcolor="white",
    updatemenus=[dict(
        type="buttons",
        buttons=buttons,
        direction="right",
        x=0.5,
        y=1.05,
        xanchor="center",
        yanchor="top"
    )],
    annotations=[
        dict(
            text="<b><i>zoom in/out with this slider</i></b>",
            x=0.5,
            y=-0.12,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(size=11, color="black")
        )
    ]
)

fig.update_layout(width=1300, height=800)

fig.show()
fig.write_html("plot1.html")

In [23]:
# Define weekend vs weekday
weekend_days = [1, 7]
weekday_days = [2, 3, 4, 5, 6]

# Prepare household dataframe
household_df = pd.concat([
    pd.DataFrame({
        "day_type": "Weekend",
        "duration": atus_actresp.loc[
            atus_actresp["TUDIARYDAY"].isin(weekend_days),
            "household_task_duration"
        ],
        "activity": "Household"
    }),
    pd.DataFrame({
        "day_type": "Weekday",
        "duration": atus_actresp.loc[
            atus_actresp["TUDIARYDAY"].isin(weekday_days),
            "household_task_duration"
        ],
        "activity": "Household"
    }),
], ignore_index=True)

household_df = household_df.dropna()
household_df["q_low"] = household_df.groupby("day_type")["duration"].transform(lambda x: x.quantile(0.01))
household_df["q_high"] = household_df.groupby("day_type")["duration"].transform(lambda x: x.quantile(0.99))
household_df["duration_capped"] = household_df.apply(lambda row: min(max(row["duration"], row["q_low"]), row["q_high"]), axis=1)
household_df["is_outlier"] = (household_df["duration"] < household_df["q_low"]) | (household_df["duration"] > household_df["q_high"])

# Prepare childcare dataframe
childcare_df = pd.concat([
    pd.DataFrame({
        "day_type": "Weekend",
        "duration": atus_actresp.loc[
            atus_actresp["TUDIARYDAY"].isin(weekend_days),
            "child_care_duration"
        ],
        "activity": "Childcare"
    }),
    pd.DataFrame({
        "day_type": "Weekday",
        "duration": atus_actresp.loc[
            atus_actresp["TUDIARYDAY"].isin(weekday_days),
            "child_care_duration"
        ],
        "activity": "Childcare"
    }),
], ignore_index=True)

childcare_df = childcare_df[(childcare_df["duration"] > 0)].dropna()
childcare_df["q_low"] = childcare_df.groupby("day_type")["duration"].transform(lambda x: x.quantile(0.01))
childcare_df["q_high"] = childcare_df.groupby("day_type")["duration"].transform(lambda x: x.quantile(0.99))
childcare_df["duration_capped"] = childcare_df.apply(lambda row: min(max(row["duration"], row["q_low"]), row["q_high"]), axis=1)
childcare_df["is_outlier"] = (childcare_df["duration"] < childcare_df["q_low"]) | (childcare_df["duration"] > childcare_df["q_high"])

# Combine both datasets
combined_df = pd.concat([household_df, childcare_df], ignore_index=True)

# Create a new x-axis variable for side-by-side violins
combined_df["day_activity"] = combined_df["day_type"] + " - " + combined_df["activity"]

# Plot all four violins side by side in the desired order
fig = px.violin(
    combined_df[~combined_df["is_outlier"]],
    x="day_activity",
    y="duration_capped",
    color="activity",
    box=True,
    points=False,
    title="Time Spent on Household Tasks vs. Childcare per Day: Weekend vs Weekday",
    labels={"duration_capped": "Sqrt Minutes per Day", "day_activity": ""},
    category_orders={"day_activity": [
        "Weekday - Household",
        "Weekend - Household",
        "Weekday - Childcare",
        "Weekend - Childcare"
    ]}
)

fig.update_traces(
    width=0.8,
    scalemode="count",
    spanmode="hard",
    hoveron="violins",
    hovertemplate="%{y:.0f} minutes<extra></extra>",
    selector=dict(type="violin")
)

# Add outliers
outliers = combined_df[combined_df["is_outlier"]]
fig.add_scatter(
    x=outliers["day_activity"],
    y=outliers["duration"],
    mode="markers",
    marker=dict(color="black", size=4),
    showlegend=False
)

# White background + grid lines
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend_title_text=None,
    xaxis=dict(
        gridcolor="lightgrey",
        zerolinecolor="lightgrey",
        showline=True,
        linecolor="lightgrey"
    ),
    yaxis=dict(
        gridcolor="lightgrey",
        zerolinecolor="lightgrey",
        showline=True,
        linecolor="lightgrey",
        showspikes=True,
        spikecolor="black",
        spikesnap="cursor",
        spikemode="across",
        spikethickness=1
    )
)

fig.show()
fig.write_html("plot2.html")

In [24]:
## Plot 3

# Filter for employed respondents
atus_employed = atus_actresp[atus_actresp["employed"] == 1].copy()

# Reshape to long format
atus_long = atus_employed.melt(
    id_vars=["TUCASEID", "TUDIARYDATE", "annual_income"],
    value_vars=["child_care_duration", "household_task_duration"],
    var_name="activity_type",
    value_name="minutes"
)

# Random sampling
child_care_sample = atus_long[atus_long["activity_type"] == "child_care_duration"].sample(n=1000, random_state=42)
household_sample = atus_long[atus_long["activity_type"] == "household_task_duration"].sample(n=1000, random_state=42)
atus_sampled = pd.concat([child_care_sample, household_sample])

# Map activity_type to readable names
atus_sampled["activity_label"] = atus_sampled["activity_type"].map({
    "child_care_duration": "Child Care",
    "household_task_duration": "Household Task"
})

# Scatter plot using the new labels
fig = px.scatter(
    atus_sampled,
    x="TUDIARYDATE",
    y="minutes",
    color="activity_label",
    color_discrete_map={"Child Care": "#183AB4", "Household Task": "#883955"},
    hover_data={"minutes": True, "activity_label": True},
    labels={
        "minutes": "Minutes Spent",
        "TUDIARYDATE": "Diary Date",
        "activity_label": "Activity"
    },
    title="Unpaid Work Time in 24 Hours for Employed Respondents"
)

fig.update_traces(marker=dict(opacity=0.6, size=6))

# Determine full x-axis range
full_start = atus_sampled["TUDIARYDATE"].min()
full_end = atus_sampled["TUDIARYDATE"].max()

# Add "All" button first
all_button = dict(
    label="All",
    method="relayout",
    args=[{"xaxis.range": [full_start, full_end]}]
)

# Add updatemenus for year buttons
years = list(range(2010, 2020))
buttons = [all_button]
for y in years:
    start = datetime(y, 1, 1)
    end = datetime(y, 12, 31)
    buttons.append(
        dict(
            label=str(y),
            method="relayout",
            args=[{"xaxis.range": [start, end]}]
        )
    )


# Update layout with buttons and range slider
fig.update_layout(
    margin=dict(b=120),
    xaxis=dict(
        title=dict(text="Diary Date", standoff=120),
        rangeslider=dict(visible=True, bgcolor="lightgrey", thickness=0.05),
        type="date",
        showgrid=True,
        gridcolor="lightgrey",
        gridwidth=1,
        zeroline=False
    ),
    yaxis=dict(
        title="Sqrt Minutes per Day",
        autorange=True,
        showgrid=True,
        gridcolor="lightgrey",
        gridwidth=1,
        zeroline=False
    ),
    dragmode="zoom",
    legend=dict(
        y=1,
        yanchor="top",
        x=1.01,
        xanchor="left"
    ),
    paper_bgcolor="white",
    plot_bgcolor="white",
    updatemenus=[dict(
        type="buttons",
        buttons=buttons,
        direction="right",
        x=0.5,
        y=1.05,
        xanchor="center",
        yanchor="top"
    )],
    annotations=[
        dict(
            text="<b><i>zoom in/out with this slider</i></b>",
            x=0.5,
            y=-0.11,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(size=11, color="black")
        )
    ]
)


fig.update_layout(width=1300, height=800)
fig.show()
fig.write_html("plot3.html")